# Gift-Eval Ablation Analysis

This notebook analyzes the effects of a TSFM's performance across different datasets.

**Ablation Levels:**
- Level 0: Original model (no ablation)
- Level 1: Ablate layers 10, heads-all
- Level 2: Ablate layers 10-11, heads-all
- Level 3: Ablate layers 10-11-12, heads-all
- Level 4: Ablate layers 10-11-12-13, heads-all
- Level 5: Ablate layers 10-11-12-13-14, heads-all
- Level 6: Ablate layers 10-11-12-13-14-15, heads-all


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from tsfm_lens.utils import apply_custom_style, normalize_by_seasonal_naive
from scipy.stats import gmean


In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
HOME_DIR = os.getenv("HOME")

In [ ]:
root_dir = os.path.join(HOME_DIR, "tsfm-lens")

In [ ]:
model_name = "TimesFM 2.5"

model_to_dir_mapping = {
    "TimesFM 2.5": "google-timesfm-2.5-200m-pytorch",
    "Chronos Bolt": "amazon-chronos-bolt-base",
    "Chronos": "amazon-chronos-t5-base",
    "Toto": "Datadog-Toto-Open-Base-1.0_samples-20",
}

In [ ]:
# Select the metric to analyze
SELECTED_METRIC = "eval_metrics/MASE[0.5]"
# SELECTED_METRIC = "eval_metrics/sMAPE[0.5]"

metric_pretty_name = "".join(c for c in SELECTED_METRIC.split("/")[-1] if c.isalpha())

print(f"Analyzing metric: {SELECTED_METRIC}")
print(f"Metric name: {metric_pretty_name}")


In [ ]:
save_figs = False
figs_save_dir = os.path.join(root_dir, "figs", "gift-eval", model_to_dir_mapping[model_name])
if not os.path.exists(figs_save_dir):
    os.makedirs(figs_save_dir)
if save_figs:
    print(f"Saving figures to: {figs_save_dir}")

## Load Data


In [ ]:
extra_description_str_by_key = {}

if model_name == "TimesFM 2.5":
    # TimesFM 2.5
    # selected_layers_lst = [0, 10, 19, [10, 11, 12, 13], [7, 8, 9, 10, 11, 12, 13]]
    # selected_layers_lst = [8, [10, 11, 12, 13], [7, 8, 9, 10, 11, 12, 13]]
    selected_layers_lst = [0, 5, 10, 15, 19]
    levels_num_heads_ablated = [1, 2, 4, 6, 8, 10, 12]
    n_divisor = 16  # number of heads in a layer

    ablated_files_dict = {}
    ablation_meaning_str_dict = {}
    ablation_level_meaning_str_dict = {}
    ablation_level_meaning_alternative_str_dict = {}

    ablation_meaning_str = "Ablation of Heads Across Layers"
    ablation_level_meaning_str = "Pct of Heads Ablated per Layer"
    ablation_level_meaning_alternative_str = "Number of Heads Ablated per Layer"

    for i, selected_layers in enumerate(selected_layers_lst):
        key = selected_layers
        if isinstance(selected_layers, list):
            selected_layers_str = "-".join(map(str, selected_layers))
            key = f"Layers {selected_layers[0]}-{selected_layers[-1]}"
            ablation_meaning_str_dict[key] = f"Ablation of Heads in Layers {key}"
            ablation_level_meaning_str_dict[key] = "Pct of Heads Ablated per Layer"
            ablation_level_meaning_alternative_str_dict[key] = "Number of Heads Ablated per Layer"

        else:
            selected_layers_str = str(selected_layers)
            key = f"Layer {selected_layers_str}"
            ablation_meaning_str_dict[key] = f"Ablation of Heads in Layer {selected_layers_str}"
            ablation_level_meaning_str_dict[key] = "Pct of Heads Ablated"
            ablation_level_meaning_alternative_str_dict[key] = "Number of Heads Ablated"

        ablation_files = {
            0: "original_all_results.csv",
            **{n: f"head_layers_{selected_layers_str}_heads-{n}_all_results.csv" for n in levels_num_heads_ablated},
            n_divisor: f"head_layers_{selected_layers_str}_heads-all_all_results.csv",
            # **{n: f"head-mlp_layers_{selected_layers_str}_heads-{n}_all_results.csv" for n in levels_num_heads_ablated},
            # n_divisor: f"head-mlp_layers_{selected_layers_str}_heads-all_all_results.csv",
        }
        ablation_files = {round(k, 3) if isinstance(k, float) else k: v for k, v in ablation_files.items()}
        ablated_files_dict[key] = ablation_files

    extra_description_str_by_key = {
        "Layer 0": "First Layer",
        "Layer 5": "Middle Layer",
        "Layer 10": "Middle Layer",
        "Layer 15": "Middle Layer",
        "Layer 19": "Last Layer",
    }

    # extra_description_str_by_key = {
    #     "Layer 0": "First Layer",
    #     # "Layer 8": "Middle Layer",
    #     "Layer 10": "Middle Layer",
    #     "Layer 19": "Last Layer",
    #     "Layers 10-13": "20% of Layers",
    #     "Layers 7-13": "35% of Layers",
    # }

elif model_name == "Chronos Bolt":
    # Chronos Bolt
    # selected_layers_lst = [0, 2, 11, [1, 2, 3, 4], [1, 2, 3, 4, 5, 6]]
    selected_layers_lst = [0, 2, 11, [1, 2, 3], [1, 2, 3, 4], [1, 2, 3, 4, 5, 6]]
    levels_num_heads_ablated = [1, 2, 4, 6, 8, 10]
    n_divisor = 12  # number of heads in a layer

    ablated_files_dict = {}
    ablation_meaning_str_dict = {}
    ablation_level_meaning_str_dict = {}
    ablation_level_meaning_alternative_str_dict = {}

    ablation_meaning_str = "Ablation of Heads Across Layers"
    ablation_level_meaning_str = "Pct of Heads Ablated per Layer"
    ablation_level_meaning_alternative_str = "Number of Heads Ablated per Layer"

    for selected_layers in selected_layers_lst:
        key = selected_layers
        if isinstance(selected_layers, list):
            selected_layers_str = "-".join(map(str, selected_layers))
            key = f"Layers {selected_layers[0]}-{selected_layers[-1]}"
            ablation_meaning_str_dict[key] = f"Ablation of Heads in Layers {key}"
            ablation_level_meaning_str_dict[key] = "Pct of Heads Ablated per Layer"
            ablation_level_meaning_alternative_str_dict[key] = "Number of Heads Ablated per Layer"

        else:
            selected_layers_str = str(selected_layers)
            key = f"Layer {selected_layers_str}"
            ablation_meaning_str_dict[key] = f"Ablation of Heads in Layer {selected_layers_str}"
            ablation_level_meaning_str_dict[key] = "Pct of Heads Ablated"
            ablation_level_meaning_alternative_str_dict[key] = "Number of Heads Ablated"

        ablation_files = {
            0: "original_all_results.csv",
            **{n: f"head_layers_{selected_layers_str}_heads-{n}_all_results.csv" for n in levels_num_heads_ablated},
            n_divisor: f"head_layers_{selected_layers_str}_heads-all_all_results.csv",
            # n_divisor: f"head-mlp_layers_{selected_layers_str}_heads-all_all_results.csv",
        }
        ablation_files = {round(k, 3) if isinstance(k, float) else k: v for k, v in ablation_files.items()}
        ablated_files_dict[key] = ablation_files

    extra_description_str_by_key = {
        "Layer 0": "First Layer",
        "Layer 2": "Middle Layer",
        "Layer 11": "Last Layer",
        "Layers 1-3": "25% of Layers",
        "Layers 1-4": "33% of Layers",
        "Layers 1-6": "50% of Layers",
    }
elif model_name == "Toto":
    # selected_layers_lst = [0, 2, 11, [9, 10, 11], [8, 9, 10, 11]]
    selected_layers_lst = [0, 2, 11, [2, 9, 10], [2, 8, 9, 10]]
    levels_num_heads_ablated = [1, 2, 4, 6, 8, 10]
    n_divisor = 12  # number of heads in a layer

    ablated_files_dict = {}
    ablation_meaning_str_dict = {}
    ablation_level_meaning_str_dict = {}
    ablation_level_meaning_alternative_str_dict = {}

    ablation_meaning_str = "Ablation of Heads Across Layers"
    ablation_level_meaning_str = "Pct of Heads Ablated per Layer"
    ablation_level_meaning_alternative_str = "Number of Heads Ablated per Layer"

    for selected_layers in selected_layers_lst:
        key = selected_layers
        if isinstance(selected_layers, list):
            selected_layers_str = "-".join(map(str, selected_layers))
            # Check if layers are contiguous
            is_contiguous = all(
                selected_layers[i] + 1 == selected_layers[i + 1] for i in range(len(selected_layers) - 1)
            )

            if is_contiguous:
                key = f"Layers {selected_layers[0]}-{selected_layers[-1]}"
            else:
                # Build segments for non-contiguous layers
                segments = []
                i = 0
                while i < len(selected_layers):
                    start = i
                    while i + 1 < len(selected_layers) and selected_layers[i + 1] == selected_layers[i] + 1:
                        i += 1
                    if start == i:
                        segments.append(str(selected_layers[start]))
                    else:
                        segments.append(f"{selected_layers[start]}-{selected_layers[i]}")
                    i += 1
                key = f"Layers {', '.join(segments)}"
            ablation_meaning_str_dict[key] = f"Ablation of Heads in Layers {key}"
            ablation_level_meaning_str_dict[key] = "Pct of Heads Ablated per Layer"
            ablation_level_meaning_alternative_str_dict[key] = "Number of Heads Ablated per Layer"

        else:
            selected_layers_str = str(selected_layers)
            key = f"Layer {selected_layers_str}"
            ablation_meaning_str_dict[key] = f"Ablation of Heads in Layer {selected_layers_str}"
            ablation_level_meaning_str_dict[key] = "Pct of Heads Ablated"
            ablation_level_meaning_alternative_str_dict[key] = "Number of Heads Ablated"

        ablation_files = {
            0: "original_all_results.csv",
            **{n: f"head_layers_{selected_layers_str}_heads-{n}_all_results.csv" for n in levels_num_heads_ablated},
            n_divisor: f"head_layers_{selected_layers_str}_heads-all_all_results.csv",
            # n_divisor: f"head-mlp_layers_{selected_layers_str}_heads-all_all_results.csv",
        }
        ablation_files = {round(k, 3) if isinstance(k, float) else k: v for k, v in ablation_files.items()}
        ablated_files_dict[key] = ablation_files

    extra_description_str_by_key = {
        "Layer 0": "First Layer",
        "Layer 2": "Middle Layer",
        "Layer 11": "Last Layer",
        "Layers 2, 9-10": "25% of Layers",
        "Layers 2, 8-10": "33% of Layers",
    }

else:
    raise ValueError(f"Model {model_name} not supported yet")

In [ ]:
ablation_files

In [ ]:
ablated_files_dict.keys()

In [ ]:
# rseeds_lst = [123, 99, 42, 688, 22, 33, 56, 78, 357, 234, 567, 890, 1356, 2233, 3566, 4891, 222, 333, 444, 555, 777]
# metrics_dir_dict = {
#     rseed: os.path.join(root_dir, "results", model_to_dir_mapping[model_name], f"rseed-{rseed}") for rseed in rseeds_lst
# }

rseeds_lst = [99, 42]
metrics_dir_dict = {
    rseed: os.path.join(root_dir, "results", model_to_dir_mapping[model_name], f"srank_reverse_rseed-{rseed}")
    for rseed in rseeds_lst
}

print(f"Metrics directory: {metrics_dir_dict}")

In [ ]:
ablated_files_dict

In [ ]:
# Load and combine CSV files across ablation levels and rseeds
# Strategy: For each level, compute geometric mean across datasets for each rseed,
# then select the rseed with the lowest geometric mean
best_filepaths_by_key = {}
data_by_key = {}
for key, ablation_files in ablated_files_dict.items():
    if key not in data_by_key:
        data_by_key[key] = {}
    if key not in best_filepaths_by_key:
        best_filepaths_by_key[key] = {}
    curr_data_frame = {}

    for level, filepath in ablation_files.items():
        # Dictionary to store data by rseed: {rseed: df}
        rseed_data = {}
        available_rseeds = []

        for rseed, metrics_dir in metrics_dir_dict.items():
            df_filepath = os.path.join(metrics_dir, filepath)
            if not os.path.exists(df_filepath):
                print(f"Level {level}, rseed {rseed}: Skipping (not found)")
                continue
            print(f"Level {level}, rseed {rseed}: Loading")
            df = pd.read_csv(df_filepath)
            # check if df has 97 rows; otherwise it is incomplete and should be skipped
            if len(df) != 97:
                print(f"Level {level}, rseed {rseed}: Skipping (incomplete)")
                continue
            df["ablation_level"] = level
            rseed_data[rseed] = (df, df_filepath)
            available_rseeds.append(rseed)
        if not rseed_data:
            print(f"Level {level}: No data found, skipping")
            continue

        # For each rseed, compute geometric mean of metrics across all datasets
        # Then select the rseed with the lowest geometric mean
        metric_cols = None
        rseed_gmeans = {}

        for rseed, (df, df_filepath) in rseed_data.items():
            if metric_cols is None:
                # Select only numeric columns that are metrics (exclude metadata columns)
                exclude_cols = [
                    "dataset",
                    "model",
                    "domain",
                    "num_variates",
                    "num_datasets",
                    "ablation_level",
                    "frequency",
                ]
                metric_cols = [
                    col for col in df.columns if col not in exclude_cols and pd.api.types.is_numeric_dtype(df[col])
                ]

            # Compute geometric mean for the selected metric only
            selected_metric_gmean = gmean(df[SELECTED_METRIC])

            # Store the geometric mean of the selected metric as the overall score
            rseed_gmeans[rseed] = {
                f"gmean_{SELECTED_METRIC}": selected_metric_gmean,
                "filepath": df_filepath,
                "df": df,
            }

        # Find rseed with lowest overall geometric mean
        best_rseed = min(rseed_gmeans.keys(), key=lambda r: rseed_gmeans[r][f"gmean_{SELECTED_METRIC}"])
        best_df = rseed_gmeans[best_rseed]["df"]
        best_filepath = rseed_gmeans[best_rseed]["filepath"]

        # Store the best filepath information
        best_filepaths_by_key[key][level] = {
            "best_rseed": best_rseed,
            "best_filepath": best_filepath,
            "available_rseeds": available_rseeds,
            f"gmean_{SELECTED_METRIC}": rseed_gmeans[best_rseed][f"gmean_{SELECTED_METRIC}"],
            "num_rseeds_evaluated": len(rseed_data),
        }

        curr_data_frame[level] = best_df
        print(
            f"Level {level}: Best rseed={best_rseed} (gmean={rseed_gmeans[best_rseed][f'gmean_{SELECTED_METRIC}']:.6f}) from {len(rseed_data)} rseeds"
        )

    all_data_for_key = pd.concat(curr_data_frame.values(), ignore_index=True)
    data_by_key[key] = all_data_for_key
    print(f"\nTotal: {len(all_data_for_key)} rows, {all_data_for_key['dataset'].nunique()} unique datasets")

In [ ]:
# Structure: best_filepaths_by_key[key][level] = {"rseed": ..., "filepath": ..., "overall_gmean": ..., "metric_gmeans": {...}}
print("Example of stored filepath information:")
print("=" * 80)
example_key = list(best_filepaths_by_key.keys())[1]
for key in [example_key]:  # Just show first key as example
    print(f"\nKey: {key}")
    for level in list(best_filepaths_by_key[key].keys()):
        info = best_filepaths_by_key[key][level]
        print(f"  Level {level}:")
        print(f"    Best rseed: {info['best_rseed']}, {info['best_filepath']}")
        print(f"    {SELECTED_METRIC} gmean: {info[f'gmean_{SELECTED_METRIC}']:.6f}")
        print(f"    {info['num_rseeds_evaluated']} rseeds evaluated: {info['available_rseeds']}")


In [ ]:
rng = np.random.default_rng(333)
layers_to_ablate = [10]
ablate_n_heads_per_layer = 10
heads_to_ablate = []
for layer in layers_to_ablate:
    heads_to_ablate_for_layer = rng.choice(list(range(12)), size=ablate_n_heads_per_layer, replace=False)
    heads_to_ablate.extend([(layer, head) for head in heads_to_ablate_for_layer])  # type: ignore
print(heads_to_ablate)

## Optional: Normalize by Seasonal Naive Baseline


In [ ]:
# Configuration: Set to True to normalize metrics by seasonal naive baseline
NORMALIZE_BY_SEASONAL_NAIVE = True

print(f"Normalization by seasonal naive baseline: {NORMALIZE_BY_SEASONAL_NAIVE}")


In [ ]:
if NORMALIZE_BY_SEASONAL_NAIVE:
    for key in data_by_key.keys():
        all_data = data_by_key[key]
        print("\nLoading seasonal naive baseline...")
        seasonal_naive_path = os.path.join(root_dir, "results", "seasonal_naive_baseline", "all_results.csv")
        seasonal_naive_df = pd.read_csv(seasonal_naive_path)
        print(f"Loaded seasonal naive baseline: {len(seasonal_naive_df)} datasets")

        print("\nNormalizing data by seasonal naive baseline...")
        all_data = normalize_by_seasonal_naive(all_data, seasonal_naive_df)
        data_by_key[key] = all_data
else:
    print("\nSkipping normalization.")

## Display Available Metrics


In [ ]:
# Display available metrics
metric_columns = [col for col in all_data.columns if "eval_metrics" in col]
print("Available metrics:")
for i, metric in enumerate(metric_columns, 1):
    print(f"{i}. {metric}")


## 1. Box Plot: Overall Performance Across Ablation Levels


In [ ]:
# Prepare data for box plot
box_data_by_key = {}
for key in data_by_key.keys():
    box_data = data_by_key[key][["dataset", "ablation_level", SELECTED_METRIC]].copy()
    # Remove any infinite or NaN values
    box_data = box_data.replace([np.inf, -np.inf], np.nan).dropna()
    box_data_by_key[key] = box_data


## 2. Line Plot: Median Performance Trend


In [ ]:
# Calculate median and percentiles for each ablation level and key
stats_by_level_by_key = {}
for key in data_by_key.keys():
    box_data = box_data_by_key[key]
    stats_by_level = (
        box_data.groupby("ablation_level")[SELECTED_METRIC]
        .agg(
            [
                "median",
                "mean",
                lambda x: gmean(x),
                lambda x: np.exp(np.std(np.log(x)) / np.sqrt(len(x))),  # geometric standard error (scale factor)
                lambda x: x.quantile(0.25),
                lambda x: x.quantile(0.75),
            ]
        )
        .rename(columns={"<lambda_0>": "geom_mean", "<lambda_1>": "geom_ste", "<lambda_2>": "q25", "<lambda_3>": "q75"})
        .reset_index()
    )
    stats_by_level_by_key[key] = stats_by_level

In [ ]:
single_layer_colors = plt.cm.get_cmap("Blues")(np.linspace(0.4, 1.0, 3))
# layergroups_colors = ["goldenrod", "darkorange"]
layergroups_colors = ["goldenrod", "darkorange", "gold"]

colors = list(single_layer_colors) + list(layergroups_colors)

In [ ]:
layergroups_colors

In [ ]:
data_by_key.keys()

In [ ]:
# Create line plot with error bands
fig, ax = plt.subplots(figsize=(6, 6))

# Plot lines for each key
for i, key in enumerate(data_by_key.keys()):
    extra_description_str = extra_description_str_by_key[key]
    stats_by_level = stats_by_level_by_key[key]

    # Get baseline values
    baseline_geom_mean = stats_by_level.loc[stats_by_level["ablation_level"] == 0, "geom_mean"].values[0]

    if i == 0:
        # Plot baseline reference lines
        ax.axhline(
            y=baseline_geom_mean,
            color="black",
            linestyle="--",
            linewidth=2,
            label=f"Baseline ({baseline_geom_mean:.3f})",
            alpha=0.7,
            zorder=4,
        )
        ax.text(
            0,
            baseline_geom_mean * 0.994,
            f"{baseline_geom_mean:.3f}",
            color="red",
            fontsize=10,
            fontweight="bold",
            va="top",
            ha="left",
            zorder=100,
        )

        ax.axhline(y=1.02 * baseline_geom_mean, color="gray", linestyle="--", linewidth=2)
        ax.text(
            0,
            (1.02 * baseline_geom_mean) * 1.004,
            "Baseline + 2%",
            color="orangered",
            fontsize=10,
            fontweight="bold",
            va="bottom",
            ha="left",
            zorder=100,
        )

    color = colors[i]
    # Plot geometric mean line with error envelope
    ax.plot(
        stats_by_level["ablation_level"] / n_divisor * 100,
        stats_by_level["geom_mean"],
        marker="^",
        markerfacecolor="None",
        markeredgecolor=color,
        markeredgewidth=2,
        linewidth=2.5,
        markersize=7,
        label=f"{key} ({extra_description_str})",
        color=color,
        linestyle="-",
        alpha=0.9,
    )

    ax.fill_between(
        stats_by_level["ablation_level"] / n_divisor * 100,
        stats_by_level["geom_mean"] * (stats_by_level["geom_ste"] ** 0.5),
        stats_by_level["geom_mean"] / (stats_by_level["geom_ste"] ** 0.5),
        alpha=0.05,
        color=color,
        zorder=-1,
    )

# Formatting
ax.set_xlabel(ablation_level_meaning_str + " (%)", fontweight="bold")
ax.set_ylabel(f"{metric_pretty_name} (Geom Mean)", fontweight="bold")
ax.set_title(f"{ablation_meaning_str} ({model_name})", fontweight="bold", pad=20)
ax.grid(True, alpha=0.2)
ax.legend(loc="upper left", frameon=True)

# Set x-axis ticks
all_levels = set()
for stats_by_level in stats_by_level_by_key.values():
    all_levels.update(stats_by_level["ablation_level"].unique())
xticks = sorted([x / n_divisor * 100 for x in all_levels])
ax.set_xticks(xticks)
ax.set_xticklabels([f"{int(x)}" if int(x) == x else f"{x:.1f}" for x in xticks], rotation=0)

# Set y-axis limits
ax.set_ylim(0.95 * baseline_geom_mean, 1.25 * baseline_geom_mean)

plt.tight_layout()
save_path = os.path.join(figs_save_dir, f"{ablation_meaning_str}_line_plot_{model_name}.pdf")
if save_figs:
    plt.savefig(save_path, bbox_inches="tight")
    print(f"Saved line plot to: {save_path}")
plt.show()

In [ ]:
# Calculate and display percentage change from baseline for each key
for key in data_by_key.keys():
    stats_by_level = stats_by_level_by_key[key]
    baseline_median = stats_by_level.loc[stats_by_level["ablation_level"] == 0, "median"].values[0]
    baseline_geom_mean = stats_by_level.loc[stats_by_level["ablation_level"] == 0, "geom_mean"].values[0]

    print(f"\nPercentage Change from Baseline (Level 0) for {key}:")
    print("=" * 70)
    for _, row in stats_by_level.iterrows():
        num_heads_ablated = int(row["ablation_level"])
        pct_heads_ablated = num_heads_ablated / n_divisor * 100

        median_change = ((row["median"] - baseline_median) / baseline_median) * 100
        curr_geom_mean = row["geom_mean"]
        geom_mean_change = ((curr_geom_mean - baseline_geom_mean) / baseline_geom_mean) * 100
        print(
            f"{num_heads_ablated} heads ({pct_heads_ablated:.1f}%) ablated: Median {median_change:+.2f}%, Geom Mean {geom_mean_change:+.2f}%"
        )
        print(f"==== Current Geom Mean: {curr_geom_mean:.3f}")
